# Aula 2, Multi-head attention

Notebook executável que acompanha a aula [02-multi-head-attention.md](../../lessons/modulo-06-transformers/02-multi-head-attention.md).

## O que vamos fazer aqui

Em vez de uma atenção só, o Transformer usa várias cabeças em paralelo, cada uma com
projeções próprias, livres para aprender relações diferentes. Vamos implementar a multi-head
attention do zero, com duas cabeças, e ver que cada uma olha a frase de um jeito. Só numpy.

## Projeções e cabeças

Projetamos os embeddings para query, key e value e fatiamos as dimensões entre as cabeças.
Cada cabeça calcula a sua própria atenção em um subespaço.

In [ ]:
import numpy as np

def softmax(z, eixo=-1):
    z = z - z.max(axis=eixo, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=eixo, keepdims=True)


tokens = ["gato", "felino", "pulou", "alto"]
# Embeddings escolhidos: gato e felino são parecidos; pulou e alto, distintos.
E = np.array([
    [1.0, 0.0, 0.0, 0.0],   # gato
    [0.9, 0.1, 0.0, 0.0],   # felino (parecido com gato)
    [0.0, 0.0, 1.0, 0.0],   # pulou
    [0.0, 0.0, 0.0, 1.0],   # alto
])

rng = np.random.default_rng(0)
d_model, n_heads = 4, 2
d_head = d_model // n_heads

Wq = rng.normal(0, 1, (d_model, d_model))
Wk = rng.normal(0, 1, (d_model, d_model))
Wv = rng.normal(0, 1, (d_model, d_model))
Q, K, V = E @ Wq, E @ Wk, E @ Wv

saidas = []
for h in range(n_heads):
    fatia = slice(h * d_head, (h + 1) * d_head)     # dimensões desta cabeça
    scores = Q[:, fatia] @ K[:, fatia].T / np.sqrt(d_head)
    A = softmax(scores, eixo=-1)
    saidas.append(A @ V[:, fatia])
    print(f"Cabeça {h}, matriz de atenção:")
    print(np.round(A, 2))

multi = np.concatenate(saidas, axis=1)
print("\nSaída multi-head, shape:", multi.shape)

As duas cabeças mostram matrizes de atenção diferentes, cada uma fruto das suas
projeções, ou seja, olham a frase de ângulos distintos. A concatenação devolve uma
representação com a dimensão original. Empilhar essa diversidade de cabeças, camada sobre
camada, é o que dá aos Transformers a sua capacidade. Para o projeto, compare uma cabeça
única com várias na mesma frase.